# 01 - Exploratory Data Analysis

My initial look at the ultra-marathon dataset before building the bronze and silver layers.
The goal is to understand size, schema, data quality issues, and what the cleaning steps will need to handle.

*Some parts and solutions of this code was found through help from Claude (Anthropic).*

In [0]:
import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F
from utils.constants import RAW_CSV_PATH
from utils.io_helpers import read_csv_from_volume

## Load the data

Reading the raw CSV from the Unity Catalog volume.
Using `inferSchema=True` for this exploration - we'll see what Spark infers and revisit schemas in silver if needed.

In [0]:
df = read_csv_from_volume(spark, RAW_CSV_PATH)

## Size and shape

How big is this dataset?

In [0]:
n_rows = df.count()
n_cols = len(df.columns)

print(f"Rows:    {n_rows:,}")
print(f"Columns: {n_cols}")

## Schema

What columns do we have and what types did Spark infer?

In [0]:
df.printSchema()

## First rows

Sanity check - does the data look like what we expect?

In [0]:
df.show(5, vertical=True, truncate=False)

## Descriptive statistics

Quick look at the numerical columns to spot weird ranges.

In [0]:
df.describe().show(vertical=True, truncate=False)

## Null counts

How many nulls per column? This drives the cleaning strategy in silver.

In [0]:
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True, truncate=False)

## Unique events

How many distinct events are in the data?

In [0]:
n_events = df.select("Event name").distinct().count()
print(f"Unique events: {n_events:,}")

## Age distribution

Using `Year of event - Year of birth` as a proxy for age at the time of the race.

In [0]:
age_df = df.withColumn(
    "age_at_event",
    F.col("Year of event") - F.col("Athlete year of birth")
)

age_df.select("age_at_event").describe().show(vertical=True, truncate=False)

## Top countries by participation

Which countries are most represented?

In [0]:
(
    df.groupBy("Athlete country")
    .count()
    .orderBy(F.col("count").desc())
    .show(15)
)

## Summary - things to handle in silver

Some stuff I noticed during EDA that needs cleaning later:

**Data issues:**
- Year of birth has weird values - min is 1193 and max is 2021. Calculated age goes from -30 to 827 which is obviously wrong. Need to filter these out in silver.
- Athlete club is null about 38% of the time. Not great but probably ok to keep, it's not a critical field.
- Country codes are mixed case (saw both `ACT` and `swe` in the data). Should make them all uppercase.
- Gender has F, M and X. Keeping all three.

**Schema issues:**
- Event distance/length is a string like "50km" - need to split the number and the unit.
- Athlete performance is also a string, either time ("4:51:39 h") or distance ("127.5 km"). Same thing, split value and unit.
- Event dates are strings like "06.01.2018", should be a real date.
- Year of birth came in as double for some reason, should be int.
- Column names have spaces and slashes which is annoying to work with, rename to snake_case.

**Validation rules (from the lab instructions):**
- km or mi event -> performance should be a time (h)
- h event -> performance should be a distance (km or mi)
- Drop rows with 'd' (days) as time unit

**Sizes for the dim tables later:**
- ~27k unique events
- ~1.6M unique athletes (a lot!)
- 15+ countries with lots of runners